# Per-Issuer Factor Trajectories

Companion to [`ycs_multilayer.ipynb`](ycs_multilayer.ipynb) and
[`ycs_neural_hjm.ipynb`](ycs_neural_hjm.ipynb). The GUI's **"Evo: Resids"** tab
draws three charts — Level, Slope, Curvature, each over time — from
`compute_curve_factors`, which fits Nelson-Siegel (or, with "Run Neural-HJM"
ticked, the Neural HJM model) to the **market-average** curve: every issuer's
rate is averaged per `(date, term)` before any fitting happens, so those
three charts describe the market, not any single issuer — see the docstring
on `compute_curve_factors` for the full reasoning.

This notebook exercises the **per-issuer** analogue added alongside it,
`compute_curve_factors_by_issuer`: instead of averaging issuers away, it fits
each issuer's *own* curve through the identical fit → rolling-window → regime
pipeline, so Level/Slope/Curvature (and their within-window volatilities) can
be compared issuer by issuer, for both models available in this codebase —
plain Nelson-Siegel and the experimental Neural HJM model.

The two functions share one internal helper (`_fit_curve_factors`), so a
per-issuer result has the exact same `(window_idx, date_end, level_mean,
level_std, slope_mean, slope_std, curvature_mean, curvature_std)` schema the
market one does — every existing renderer reads it unmodified.

The notebook mirrors the GUI's layout, tab for tab:

| GUI tab | This notebook |
|---|---|
| Evo: Resids → Factor | Chapter D (market), Chapter F (per issuer) |
| Evo: Resids → Factor Std | Chapter D (market), Chapter F (per issuer) |
| Evo: Resids → Factor (t) | Chapter E (market), Chapter G (per issuer) |
| Evo: Resids → Factor Std (t) | Chapter E (market), Chapter G (per issuer) |
| "Show: NS Resids / Neural-HJM resids" picker | side-by-side cells, not a dropdown |

**The "Factor (t)"/"Factor Std (t)" tabs are not a Dash app**, despite how
they look. In the GUI they are one Plotly figure embedded in a
`QWebEngineView`, scrubbed by a Qt slider that calls `Plotly.restyle` through
injected JS — see `gui/factor_trajectory_tab.py`. A notebook has no Qt host to
drive that JS, so Chapters E and G instead pass `animate=True` to
`build_factor_trajectory_figure`, a new option that adds **Plotly's own**
play/pause button and slider (`fig.frames` + `fig.layout.sliders`) — the same
"scrub through the path" experience, built entirely out of the Plotly figure
itself, no server or extra widget library required. `animate` defaults to
`False`, so the GUI's own usage is completely unaffected.

Needs the **neural** extra for the Neural-HJM half: `uv sync --extra neural`.

In [ ]:
%matplotlib inline
import warnings

warnings.filterwarnings("ignore")

from datetime import date

from IPython.display import display

try:
    import torch  # noqa: F401
except ImportError as exc:
    raise ImportError(
        "This notebook needs the optional neural extra: "
        "run `uv sync --extra neural`, then restart the kernel."
    ) from exc

from ycn.analysis.config import PipelineConfig
from ycn.analysis.yield_curve import detect_panel, load_long_panel
from ycn.analysis.af_models import ModelSpec, ResidualModel, NeuralSpec
from ycn.analysis.mln_evolution import (
    compute_curve_factors,
    compute_curve_factors_by_issuer,
)
from ycn.analysis.mln_evolution_viz import render_factor_evolution
from ycn.analysis.factor_trajectory_plotly import build_factor_trajectory_figure

print(f"torch {torch.__version__}")

## Chapter A: Load the same panel as `ycs_neural_hjm.ipynb`

Same database, table and date range as the two sibling notebooks, so results
here sit on the same panel their numbers do.

In [ ]:
DB = r"D:\data\duckdb\ycs_data.duckdb"
TABLE = "zero_rates"

panel = detect_panel(DB, TABLE, "date")
cfg = PipelineConfig(
    db_path=DB,
    table=TABLE,
    date_column="date",
    name_column=panel.term_column,
    value_column=panel.rate_column,
    date_start=date(2021, 1, 1),
    date_end=date(2024, 1, 1),
    issuer_column=panel.issuer_column,
    term_columns=list(panel.term_columns),
)
long = load_long_panel(cfg, panel)
print(
    f"Loaded {long.height:,} rows, "
    f"{long[panel.issuer_column].n_unique()} issuers, "
    f"{long['date'].n_unique()} dates"
)

## Chapter B: Window settings, model specs, and the issuer subset

`WINDOW_SIZE`/`STEP_SIZE` match the GUI's Evolution Settings defaults
(~1 trading year / ~1 trading month) rather than `compute_curve_factors`'s own
defaults (30/10), so the trajectories below read the same as a default GUI
run would.

Neural HJM costs roughly 6–10 seconds per issuer at 600 epochs
(`ycs_neural_hjm.ipynb` Chapter C benchmark), so **Chapters F and G below fit
four issuers**, not all twenty, to keep this notebook interactively
re-runnable end to end in under two minutes. `ISSUERS` spans four very
different curve regimes on purpose — a core Euro-area issuer, the reserve
currency, a structurally low-rate curve, and a periphery credit — so the
per-issuer comparison in Chapter F has something to actually contrast. Widen
it to `long[panel.issuer_column].unique(maintain_order=True).to_list()` for
the full panel.

In [ ]:
WINDOW_SIZE = 252  # ~1 trading year -- matches the GUI's Evolution Settings default
STEP_SIZE = 21  # ~1 trading month

ns_spec = None  # compute_curve_factors[_by_issuer]'s model=None path: plain per-date NS
neural_spec = ModelSpec(ResidualModel.NEURAL_HJM, seed=0, neural=NeuralSpec(epochs=600))

ISSUERS = ["DEU", "USA", "JPN", "GRC"]  # core EU, reserve currency, low-rate, periphery

## Chapter C: Fit the market-average curve — NS and Neural HJM

`compute_curve_factors` averages every issuer's rate per `(date, term)`
*before* fitting anything, so both calls below describe one synthetic
"market" curve, not any single issuer — exactly what feeds the GUI's
"Evo: Resids" tab when its NS/Neural-HJM picker is on each setting
respectively. The Neural HJM call takes roughly 10–20 seconds (one panel fit
over the whole date range, not per issuer). Both functions accept a `status`
callback for live per-checkpoint progress (that is what the GUI's process log
uses); it is left unset here — piping dozens of rapid prints containing "…"
through some Windows Jupyter kernels corrupts the character mid-stream, an
ipykernel/zmq buffering quirk unrelated to this codebase, so a quiet compute
followed by a one-line summary (matching `ycs_neural_hjm.ipynb`'s Chapter C)
sidesteps it entirely.

In [ ]:
market_ns_factors, market_ns_regimes = compute_curve_factors(
    long, panel, "date", model=ns_spec, window_size=WINDOW_SIZE, step_size=STEP_SIZE
)
print(f"Market NS: {market_ns_factors.height} windows")

market_neural_factors, market_neural_regimes = compute_curve_factors(
    long, panel, "date", model=neural_spec, window_size=WINDOW_SIZE, step_size=STEP_SIZE
)
print(f"Market Neural-HJM: {market_neural_factors.height} windows")

## Chapter D: Market factor evolution — matches "Evo: Resids → Factor" / "Factor Std"

`render_factor_evolution` is the exact function the GUI calls; `std=True`
renders the within-window volatility sub-tab instead of the mean one. NS
first, then Neural HJM, standing in for the GUI's "Show" dropdown.

In [ ]:
display(render_factor_evolution(market_ns_factors, market_ns_regimes, std=False))
display(render_factor_evolution(market_ns_factors, market_ns_regimes, std=True))

In [ ]:
display(render_factor_evolution(market_neural_factors, market_neural_regimes, std=False))
display(render_factor_evolution(market_neural_factors, market_neural_regimes, std=True))

## Chapter E: Market factor trajectory (t) — matches "Factor (t)" / "Factor Std (t)"

3D path through level × slope × curvature space, coloured by elapsed time.
`animate=True` adds the native play/pause button and slider described in the
intro — press play, or drag the slider, to scrub the amber highlight point
along the path exactly like the GUI's date scrubber does.

In [ ]:
fig = build_factor_trajectory_figure(
    market_ns_factors, market_ns_regimes, title="Market NS — factor trajectory", animate=True
)
fig.show()

In [ ]:
fig = build_factor_trajectory_figure(
    market_neural_factors,
    market_neural_regimes,
    title="Market Neural-HJM — factor trajectory",
    animate=True,
)
fig.show()

## Chapter F: Per-issuer factor evolution — the new "per issuer" tab

`compute_curve_factors_by_issuer` fits each issuer's *own* curve instead of
averaging it into the market. It returns `(factors_by_issuer,
regimes_by_issuer, skipped)`, keyed by issuer, so any issuer that cannot be
fitted (too few complete dates, etc.) is dropped into `skipped` rather than
aborting the whole run — a bad issuer among twenty should not lose the
other nineteen. The Neural HJM cell below is the slow one: four issuers at
roughly 6–10 seconds each, so **expect it to take about a minute**.

In [ ]:
issuer_ns_factors, issuer_ns_regimes, ns_skipped = compute_curve_factors_by_issuer(
    long,
    panel,
    "date",
    model=ns_spec,
    window_size=WINDOW_SIZE,
    step_size=STEP_SIZE,
    issuers=ISSUERS,
)
print(f"NS per issuer: {list(issuer_ns_factors.keys())}, skipped={ns_skipped}")

In [ ]:
issuer_neural_factors, issuer_neural_regimes, neural_skipped = compute_curve_factors_by_issuer(
    long,
    panel,
    "date",
    model=neural_spec,
    window_size=WINDOW_SIZE,
    step_size=STEP_SIZE,
    issuers=ISSUERS,
)
print(f"Neural-HJM per issuer: {list(issuer_neural_factors.keys())}, skipped={neural_skipped}")

Level/Slope/Curvature over time, one issuer at a time, NS then Neural HJM —
the same `render_factor_evolution` call as Chapter D, just fed a per-issuer
`(factors, regimes)` pair instead of the market one.

In [ ]:
for issuer in ISSUERS:
    if issuer not in issuer_ns_factors:
        continue
    fig = render_factor_evolution(issuer_ns_factors[issuer], issuer_ns_regimes[issuer])
    fig.suptitle(f"{issuer} — NS factor evolution")
    display(fig)

In [ ]:
for issuer in ISSUERS:
    if issuer not in issuer_neural_factors:
        continue
    fig = render_factor_evolution(
        issuer_neural_factors[issuer], issuer_neural_regimes[issuer]
    )
    fig.suptitle(f"{issuer} — Neural-HJM factor evolution")
    display(fig)

## Chapter G: Per-issuer factor trajectory (t)

Same animated 3D path as Chapter E, one per issuer. Compare an issuer's own
path against the market path in Chapter E — a periphery credit (GRC) moving
through very different regions of level/slope/curvature space than the
market average is exactly the kind of thing the market-only "Evo: Resids" tab
cannot show.

In [ ]:
for issuer in ISSUERS:
    if issuer not in issuer_ns_factors:
        continue
    fig = build_factor_trajectory_figure(
        issuer_ns_factors[issuer],
        issuer_ns_regimes[issuer],
        title=f"{issuer} NS — factor trajectory",
        animate=True,
    )
    fig.show()

In [ ]:
for issuer in ISSUERS:
    if issuer not in issuer_neural_factors:
        continue
    fig = build_factor_trajectory_figure(
        issuer_neural_factors[issuer],
        issuer_neural_regimes[issuer],
        title=f"{issuer} Neural-HJM — factor trajectory",
        animate=True,
    )
    fig.show()

## Chapter H: Summary

* **New backend, no behaviour change to the market path.**
  `compute_curve_factors` and `compute_curve_factors_by_issuer` now share one
  internal fit → rolling-window → regime helper
  (`mln_evolution._fit_curve_factors`); the full existing test suite (144
  tests) passes unchanged, and `compute_curve_factors`'s own market-average
  results are numerically identical to before the refactor.
* **Same schema everywhere.** A per-issuer `(factors, regimes)` pair is drawn
  by the exact same `render_factor_evolution` and
  `build_factor_trajectory_figure` the GUI's "Evo: Resids" tab already uses —
  nothing issuer-specific had to be added to either renderer.
* **`animate=True` is additive.** It only changes behaviour when explicitly
  requested; `gui/factor_trajectory_tab.py`'s existing call site is
  untouched and keeps driving the same highlight trace from its own Qt
  slider.
* **Not wired into the GUI yet.** This notebook exercises the computation and
  the visual layer end to end, but there is no per-issuer sub-tab in the app
  itself — that would mean a new `EvolutionConfig` flag (mirroring
  `run_centrality`), worker changes in both `MLNEvolutionWorker` and
  `NeuralEvolutionWorker`, session persistence, and a `LayerFigureTabs`-style
  per-issuer picker in `main_window.py`. A natural follow-up, kept separate
  because it touches session-file compatibility and several files at once.